# Бие даалт 1: Цифр таних модель сургах
Зорилго: Гар бичмэл цифрийн зургууд бүхий dataset ашиглан модель сургах, шинэ өгөгдөл дээр ашиглах 

Ашиглах алгоритм: CNN Kenas

# 1: Шаардлагатай хэрэгслүүдээ import хийх

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
%matplotlib inline

np.random.seed(3)

from sklearn.model_selection import train_test_split #Data split method
from sklearn.metrics import confusion_matrix # Confusion matrix
import itertools

from tensorflow.keras.utils import to_categorical # Convert to one-hot-encoding
from keras.models import Sequential # Keras building base
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D # Keras layers
from keras.optimizers import RMSprop # Optimizer To train fast
from tensorflow.keras.preprocessing.image import ImageDataGenerator # Generates image patches for training
from keras.callbacks import ReduceLROnPlateau # Reduces learning rate when metric stops improving to fine tune training


sns.set(style='white', context='notebook', palette='deep') # Seaborne configs

# 2. Өгөгдөл бэлтгэх
## 2.1 Өгөгдөл load хийх

In [ ]:
# Load the data
train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")
Y_train = train["label"]

# 'Label' баганыг хасах
X_train = train.drop(labels = ["label"],axis = 1) 

# Өгөгдөл хэр тэнцвэртэй байгааг харах
g = sns.countplot(x=Y_train)

## 2.2 Null утгатай эсэхийг шалгах

In [ ]:
X_train.isnull().any().describe()
test.isnull().any().describe()

## 2.3 Нормалчлал [0-1]

In [ ]:
X_train = X_train / 255.0
test = test / 255.0

## 2.4 Хэлбэржүүлэлт

In [ ]:
# h = 28px, w = 28px , canal = 1)
X_train = X_train.values.reshape(-1,28,28,1) 
test = test.values.reshape(-1,28,28,1)

## 2.5 One hot encoding

In [ ]:
Y_train = to_categorical(Y_train, num_classes = 10)

## 2.6 Өгөгдлөө train / test хуваах

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size = 0.1)

## 2.7 Жишээ

In [ ]:
# Some examples
g = plt.imshow(X_train[0][:,:,0])

# 3. CNN
## 3.1 Модель тохируулах

In [ ]:
# Base 
model = Sequential()

# Convolutonal layer #1
model.add(Conv2D(filters = 32, kernel_size = (5,5),padding = 'Same', 
                 activation ='relu', input_shape = (28,28,1)))
# Convolutonal layer #2
model.add(Conv2D(filters = 32, kernel_size = (5,5),padding = 'Same', 
                 activation ='relu'))
# Pooling layer #1
model.add(MaxPool2D(pool_size=(2,2)))
# Dropout 
model.add(Dropout(0.25))


# Convolutional layer #3
model.add(Conv2D(filters = 64, kernel_size = (3,3),padding = 'Same', 
                 activation ='relu'))
# Convolutional layer #4
model.add(Conv2D(filters = 64, kernel_size = (3,3),padding = 'Same', 
                 activation ='relu'))
# Pooling layer #2
model.add(MaxPool2D(pool_size=(2,2), strides=(2,2)))
# Dropout
model.add(Dropout(0.25))


# Flattening to 1D
model.add(Flatten())
# Network with 256 neurons
model.add(Dense(256, activation = "relu"))
# Dropout
model.add(Dropout(0.5))
# Flattening to 1D
model.add(Dense(10, activation = "softmax"))

## 3.2 Optimizer, Loss function, Metrics

Once our layers are added to the model, we need to set up a score function, a loss function and an optimisation algorithm.

We define the loss function to measure how poorly our model performs on images with known labels. It is the error rate between the oberved labels and the predicted ones. We use a specific form for categorical classifications (>2 classes) called the "categorical_crossentropy".

The most important function is the optimizer. This function will iteratively improve parameters (filters kernel values, weights and bias of neurons ...) in order to minimise the loss. 

I choosed RMSprop (with default values), it is a very effective optimizer. The RMSProp update adjusts the Adagrad method in a very simple way in an attempt to reduce its aggressive, monotonically decreasing learning rate.
We could also have used Stochastic Gradient Descent ('sgd') optimizer, but it is slower than RMSprop.

The metric function "accuracy" is used is to evaluate the performance our model.
This metric function is similar to the loss function, except that the results from the metric evaluation are not used when training the model (only for evaluation).

In [ ]:
# We define the loss function to measure how poorly our model performs on images with known labels.


# Define the optimizer (that minimizes the loss)
optimizer = RMSprop(learning_rate=0.001, rho=0.9, epsilon=1e-08, decay=0.0)

# Compile the model
model.compile(optimizer = optimizer , loss = "categorical_crossentropy", metrics=["accuracy"])

## 3.3 Өгөгдөл хувиргалт

In [ ]:
datagen = ImageDataGenerator(
        featurewise_center=False,  # set input mean to 0 over the dataset
        samplewise_center=False,  # set each sample mean to 0
        featurewise_std_normalization=False,  # divide inputs by std of the dataset
        samplewise_std_normalization=False,  # divide each input by its std
        zca_whitening=False,  # apply ZCA whitening
        rotation_range=10,  # randomly rotate images in the range (degrees, 0 to 180)
        zoom_range = 0.1, # Randomly zoom image 
        width_shift_range=0.1,  # randomly shift images horizontally (fraction of total width)
        height_shift_range=0.1,  # randomly shift images vertically (fraction of total height)
        horizontal_flip=False,  # randomly flip images
        vertical_flip=False)  # randomly flip images


datagen.fit(X_train) # Өгөгдөл бүрийг random хувиргалтаар өөрчилснөөр overfitting багасна (Шинэ өгөгдөл таних чадвар ихэснэ)

In [ ]:
epochs = 3 
batch_size = 86

# Fit the model
history = model.fit(datagen.flow(X_train,Y_train, batch_size=batch_size),
                              epochs = epochs, validation_data = (X_val,Y_val),
                              verbose = 2, steps_per_epoch=X_train.shape[0] // batch_size)

# 4. Моделоо дүгнэх

## 4.1 Confusion matrix

In [ ]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

# Predict the values from the validation dataset
Y_pred = model.predict(X_val)
# Convert predictions classes to one hot vectors 
Y_pred_classes = np.argmax(Y_pred,axis = 1) 
# Convert validation observations to one hot vectors
Y_true = np.argmax(Y_val,axis = 1) 
# compute the confusion matrix
confusion_mtx = confusion_matrix(Y_true, Y_pred_classes) 
# plot the confusion matrix
plot_confusion_matrix(confusion_mtx, classes = range(10)) 

## 4.2 Алдаа харуулах

In [ ]:
# Display some error results 

# Errors are difference between predicted labels and true labels
errors = (Y_pred_classes - Y_true != 0)

Y_pred_classes_errors = Y_pred_classes[errors]
Y_pred_errors = Y_pred[errors]
Y_true_errors = Y_true[errors]
X_val_errors = X_val[errors]

def display_errors(errors_index,img_errors,pred_errors, obs_errors):
    """ This function shows 6 images with their predicted and real labels"""
    n = 0
    nrows = 2
    ncols = 3
    fig, ax = plt.subplots(nrows,ncols,sharex=True,sharey=True)
    for row in range(nrows):
        for col in range(ncols):
            error = errors_index[n]
            ax[row,col].imshow((img_errors[error]).reshape((28,28)))
            ax[row,col].set_title("Predicted label :{}\nTrue label :{}".format(pred_errors[error],obs_errors[error]))
            n += 1

# Probabilities of the wrong predicted numbers
Y_pred_errors_prob = np.max(Y_pred_errors,axis = 1)

# Predicted probabilities of the true values in the error set
true_prob_errors = np.diagonal(np.take(Y_pred_errors, Y_true_errors, axis=1))

# Difference between the probability of the predicted label and the true label
delta_pred_true_errors = Y_pred_errors_prob - true_prob_errors

# Sorted list of the delta prob errors
sorted_dela_errors = np.argsort(delta_pred_true_errors)

# Top 6 errors 
most_important_errors = sorted_dela_errors[-6:]

# Show the top 6 errors
display_errors(most_important_errors, X_val_errors, Y_pred_classes_errors, Y_true_errors)

In [ ]:
# predict results
results = model.predict(test)

# select the indix with the maximum probability
results = np.argmax(results,axis = 1)

results = pd.Series(results,name="Label")

In [ ]:
model.save('digit_model_by_ganbold.h5')

# 5. Бодит өгөгдөл дээр туршилт хийх

## 5.1 Өгөгдөл цэвэрлэлт

In [ ]:
import os
from PIL import Image

# Folder containing images
folder = "/kaggle/input/input-real-trust-me/Input images"

data = []
labels = []

# Sort files alphabetically to keep order consistent
for file in sorted(os.listdir(folder)):
    if file.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
        # Try to extract label from filename (before underscore or first character)
        try:
            label = int(file.split('_')[0])
        except:
            label = int(file[0])  # fallback: use first char if it's a number
        labels.append(label)

        # Load, grayscale, resize, and flatten
        img = Image.open(os.path.join(folder, file)).convert("L").resize((28, 28))
        data.append(list(img.getdata()))

# Create DataFrame and add labels
fixed_data = pd.DataFrame(data, columns=[f"pixel{i}" for i in range(28*28)])
fixed_data["label"] = labels
print(fixed_data)
# Save CSV
fixed_data.to_csv("/kaggle/working/clean_images_with_labels.csv", index=False)

## 5.2 Predict

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import load_model

#model = load_model('/kaggle/working/digit_model_by_ganbold.h5')

# Load data
y_true = fixed_data['label'].astype(int).to_numpy()
data = fixed_data.drop(columns=["label"], axis=1).to_numpy()

# Normalize & reshape
data = data / 255.0
data = 1 - data
data = data.reshape(-1, 28, 28, 1)

# Predict
pred_probs = model.predict(data)
pred_labels = np.argmax(pred_probs, axis=1)

## 5.3 Confusion Matrix / Result

In [ ]:
results_df = pd.DataFrame({
    "true_label": y_true,
    "predicted_label": pred_labels
})
results_df.to_csv("/kaggle/working/predictions_vs_true.csv", index=False)

all_labels = np.arange(10)

# Compute confusion matrix using all labels (ensures axes 0-9)
cm = confusion_matrix(y_true, pred_labels, labels=all_labels)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=all_labels)
disp.plot(cmap='Blues', xticks_rotation='vertical')
plt.title("Confusion Matrix (Digits 0-9)")
plt.show()